# eval

> Asking a model a question you already know the answer to.

Four helpers for using a model as a component rather than a conversation: pick a label, fill a schema, grade an answer, check a known question. All of them run stateless through `oneshot`, so a check never leaks into the conversation it was run from.

In [ ]:
#| default_exp eval

In [ ]:
#| export
from dataclasses import is_dataclass, fields
from typing import get_type_hints
from fastcore.all import first, listify, patch, AttrDict
from urai.chat import Chat

In [ ]:
#| hide
import json
from fastcore.test import test_eq, test_fail
from urai.opts import RUNTIMES, Runtime, register_runtime

## Reading an answer out

A model asked for a fenced answer usually gives one, and sometimes writes a paragraph around it. `extract_fence` takes the last fence of the right kind, and falls back to the whole reply when there is none — which is the right answer for a model that simply answered plainly.

In [ ]:
#| export
import re

def extract_fence(text, tag='answer'):
    "Contents of the last ```<tag> fence in `text`, else the whole stripped text."
    ms = re.findall(rf'^```{tag}[ \t]*\n(.*?)\n```', text or '', re.DOTALL | re.MULTILINE)
    return ms[-1] if ms else (text or '').strip()

def matches_(actual, expected):
    "Does `actual` contain any of `expected` (a scalar, or a list of accepted values)?"
    return any(str(e) in (actual or '') for e in listify(expected))

In [ ]:
test_eq(extract_fence('thinking\n```answer\n42\n```'), '42')
test_eq(extract_fence('```answer\nfirst\n```\nand\n```answer\nsecond\n```'), 'second')
test_eq(extract_fence('  just prose  '), 'just prose')      # no fence: the reply is the answer
test_eq(extract_fence('```sql\nSELECT 1\n```', 'sql'), 'SELECT 1')
test_eq(extract_fence(None), '')

In [ ]:
test_eq(matches_('the answer is 42', 42), True)
test_eq(matches_('the answer is 42', ['41', '42']), True)   # any accepted value will do
test_eq(matches_('the answer is 42', 43), False)
test_eq(matches_(None, 1), False)

## A chat to ask

Everything below is one `oneshot` away from a model. `_EchoChat` replies from a script instead, so the helpers can be tested for what they do with an answer rather than for what a model says.

In [ ]:
class _EchoChat(Chat):
    "A chat whose `oneshot` replies from a script, for testing what the helpers do with an answer."
    _runtime, ctx_limit, token_count = 'echo', 8192, 0
    def __init__(self, model=None, *, replies=(), **kw):
        self.replies, self.asked = list(replies), []
        self._setup(model, None)
    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        self.asked.append((prompt, sp))
        return self.replies.pop(0) if self.replies else ''
    def _structured_call(self, prompt, schema, sp):
        return json.loads(self._oneshot(prompt, sp))

register_runtime(Runtime('echo', _EchoChat, ('echo-',)))

## Classifying

The reply is matched loosely: a model told to answer with one word often answers with a sentence, and refusing that would fail a classification the model actually got right. When nothing matches, the reply comes back as it was, so the caller can see what went wrong instead of a silent `None`.

In [ ]:
#| export
@patch
def classify(self:Chat, text, labels, sp='Reply with only the single best label and nothing else.'):
    "One-shot label for `text`, run stateless so it does not touch the conversation."
    out = self.oneshot(f"{text}\n\nChoose exactly one label from: {', '.join(labels)}.",
                       sp, think=False).lower()
    return first(labels, lambda l: l.lower() in out) or out.strip()

In [ ]:
c = _EchoChat(replies=['spam'])
test_eq(c.classify('buy now', ['spam', 'ham']), 'spam')
assert 'spam, ham' in c.asked[0][0]        # the labels were offered

In [ ]:
c = _EchoChat(replies=['I think this one is Ham, personally.'])
test_eq(c.classify('hello', ['spam', 'ham']), 'ham')     # matched loosely, and cased back
c = _EchoChat(replies=['no idea'])
test_eq(c.classify('hello', ['spam', 'ham']), 'no idea') # nothing matched: say so, do not guess

## Structured output

`schema` is any callable that takes keyword arguments; a dataclass is the usual one. Nested dataclasses are rebuilt from the nested JSON, which is the part hand-rolled `schema(**d)` gets wrong.

In [ ]:
#| export
def mk_obj(schema, d):
    "Build `schema` from json-decoded `d`, rebuilding nested dataclasses. Non-dicts pass through."
    if not isinstance(d, dict): return d
    if is_dataclass(schema):
        hints = get_type_hints(schema)
        names = {f.name for f in fields(schema) if f.init}
        return schema(**{k: (mk_obj(hints[k], v) if k in hints else v)
                         for k, v in d.items() if k in names})
    return schema(**d)

@patch
def structured(self:Chat, prompt, schema, sp='Reply with only a JSON object matching the schema.'):
    "One-shot structured output. Returns `schema(...)`, rebuilding nested dataclasses."
    return mk_obj(schema, self._structured_call(prompt, schema, sp))

@patch
def _structured_call(self:Chat, prompt, schema, sp):
    "One structured completion, as json-decoded data. Every backend implements it."
    raise NotImplementedError

In [ ]:
from dataclasses import dataclass

@dataclass
class Addr: city: str; zip: str
@dataclass
class Person: name: str; addr: Addr

p = mk_obj(Person, {'name': 'Ada', 'addr': {'city': 'London', 'zip': 'N1'}})
test_eq(p, Person('Ada', Addr('London', 'N1')))
test_eq(type(p.addr), Addr)              # rebuilt, not left as a dict

In [ ]:
test_eq(mk_obj(Addr, {'city': 'Rome', 'zip': '00100', 'extra': 'ignored'}), Addr('Rome', '00100'))
test_eq(mk_obj(Person, 'not a dict'), 'not a dict')
test_eq(mk_obj(dict, {'a': 1}), {'a': 1})

In [ ]:
c = _EchoChat(replies=['{"city": "Rome", "zip": "00100"}'])
test_eq(c.structured('where?', Addr), Addr('Rome', '00100'))
test_fail(lambda: Chat('echo-1', runtime='echo')._structured_call('x', Addr, ''))

## Grading

`check` asks a question the caller already knows the answer to, and grades what comes back. The default grader is substring containment, which is enough for a number or a name. `llm_judge=True` spends a second model call instead, and `judge=` sends that call to a different model — which is the honest way to grade, since a model marking its own homework agrees with itself.

In [ ]:
#| export
#: System prompt for `check`: asks for a fenced final answer.
qa_sp_ = 'Answer the question, then put your final answer inside a ```answer fence.'

@patch
def grades(self:Chat, question, expected, actual):
    "LLM-as-judge on this chat's model: is `actual` right, given reference `expected`?"
    q = (f'Question: {question}\nReference answer: {expected}\nCandidate answer: {actual}\n'
         "Reply 'yes' if the candidate is correct, otherwise 'no'.")
    return self.classify(q, ['yes', 'no'],
                         sp="You are a strict grader. Reply with only 'yes' or 'no'.") == 'yes'

@patch
def check(self:Chat, question, expected, grade_fn=matches_, llm_judge=False, judge=None,
          tag='answer', sp=qa_sp_):
    "Ask `question` stateless, read the ```<tag> answer, and grade it against `expected`."
    a = extract_fence(self.oneshot(question, sp), tag)
    ok = (judge or self).grades(question, expected, a) if (llm_judge or judge) else grade_fn(a, expected)
    return AttrDict(question=question, expected=expected, answer=a, ok=ok)

In [ ]:
c = _EchoChat(replies=['Adding them up.\n```answer\n42\n```'])
r = c.check('what is 40+2?', 42)
test_eq((r.answer, r.ok), ('42', True))
test_eq(r.question, 'what is 40+2?')

In [ ]:
c = _EchoChat(replies=['```answer\n41\n```'])
test_eq(c.check('what is 40+2?', 42).ok, False)
test_eq(len(c.asked), 1)                 # no second call: substring grading is free

In [ ]:
# a judge is a second model call, and can be a different model entirely
answerer = _EchoChat(replies=['```answer\nforty-two\n```'])
judge = _EchoChat(replies=['yes'])
r = answerer.check('what is 40+2?', 42, judge=judge)
test_eq((r.answer, r.ok), ('forty-two', True))   # substring grading would have failed this
test_eq(len(judge.asked), 1)

In [ ]:
answerer = _EchoChat(replies=['```answer\nseven\n```'])
test_eq(answerer.check('what is 40+2?', 42, judge=_EchoChat(replies=['no'])).ok, False)
c = _EchoChat(replies=['```answer\n42\n```', 'yes'])
test_eq(c.check('what is 40+2?', 42, llm_judge=True).ok, True)   # judged by itself

In [ ]:
#| hide
del RUNTIMES['echo']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()